# Computing IQB Scores

This notebook shows how to use the **`iqb` Python library** to compute the Internet Quality Barometer composite score and explore what it reveals about internet quality for different use cases and geographies.

## What is the IQB Score?

The IQB score is a **0–1 composite** that measures how well a connection serves six everyday use cases:

| Use Case | Key requirements |
|----------|------------------|
| Web browsing | Low latency, moderate download |
| Video streaming | Higher download, low packet loss |
| Audio streaming | Modest download, low packet loss |
| Gaming | Very low latency, low loss |
| Video conferencing | Symmetric speed, low latency and loss |
| Online backup | High upload speed |

Each use case defines pass/fail thresholds for download speed, upload speed, latency, and packet loss. The IQB score aggregates these into a single number. **Higher = better quality.**

> **Input data:** IQB scores are computed from **Monthly Stats** — the pre-computed monthly parquet files described in [01-exploring-iqb-data.ipynb](01-exploring-iqb-data.ipynb). The `iqb` library fetches those same files and applies the IQB scoring formula on top of them.

## Prerequisites

Install the IQB library from the [m-lab/iqb](https://github.com/m-lab/iqb) repository:

```bash
# Using uv (recommended)
git clone https://github.com/m-lab/iqb.git
cd iqb
uv sync --dev
uv run jupyter notebook

# Or install the library directly
pip install git+https://github.com/m-lab/iqb.git#subdirectory=library
```

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import requests
from io import BytesIO
from pathlib import Path

try:
    from iqb import (
        IQBCalculator,
        IQB_CONFIG,
        IQB_DEFAULT_CONFIG,
        IQBConfig,
        IQBConfigUseCase,
        IQBConfigNetworkRequirement,
    )
    from iqb.cache.mlab import MLabDataFramePair
except ImportError as e:
    print(f"Note: {e}")
    print("  Install with: uv add 'git+https://github.com/m-lab/iqb.git#subdirectory=library'")

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", palette="muted")
except ImportError as e:
    print(f"Note: {e} — install with: uv add seaborn")

plt.rcParams["figure.figsize"] = (12, 5)


In [ ]:
# ── Discover available months ─────────────────────────────────────────────────
MANIFEST_URL = "https://measurementlab.net/data/iqb/manifest.json"
resp = requests.get(MANIFEST_URL, timeout=30)
resp.raise_for_status()

records = []
for path, meta in resp.json()["files"].items():
    parts = path.split("/")
    if len(parts) >= 6 and parts[5] == "data.parquet":
        records.append({
            "start":      pd.to_datetime(parts[2], format="%Y%m%dT%H%M%SZ"),
            "slice":      parts[4],
            "url":        meta["url"],
            "cache_path": path,
        })

catalog = (
    pd.DataFrame(records)
    .sort_values(["slice", "start"])
    .reset_index(drop=True)
)
print(f"Catalog: {catalog['slice'].nunique()} slices, "
      f"{catalog['start'].min().date()} → {catalog['start'].max().date()}")


In [ ]:
# ── Data loader ──────────────────────────────────────────────────────────────
_mem_cache: dict = {}

def load_parquet(slice_name: str, start: str) -> pd.DataFrame:
    key = (slice_name, start)
    if key in _mem_cache:
        return _mem_cache[key]
    start_ts = pd.to_datetime(start)
    row = catalog[(catalog["slice"] == slice_name) & (catalog["start"] == start_ts)]
    if row.empty:
        raise ValueError(f"No data for slice='{slice_name}', month='{start}'")
    row = row.iloc[0]
    local_path = Path(row["cache_path"])
    if local_path.exists():
        df = pd.read_parquet(local_path)
    else:
        print(f"[download] {slice_name} / {start} …")
        r = requests.get(row["url"], timeout=60)
        r.raise_for_status()
        local_path.parent.mkdir(parents=True, exist_ok=True)
        local_path.write_bytes(r.content)
        df = pd.read_parquet(BytesIO(r.content))
        print(f"  ✓ {len(df):,} rows")
    _mem_cache[key] = df
    return df

def get_country_pair(start: str, country_code: str | None = None) -> MLabDataFramePair:
    """Return a MLabDataFramePair for country-level data (optionally filtered)."""
    dl = load_parquet("downloads_by_country", start)
    ul = load_parquet("uploads_by_country", start)
    if country_code:
        dl = dl[dl["country_code"] == country_code]
        ul = ul[ul["country_code"] == country_code]
    return MLabDataFramePair(download=dl, upload=ul)

def get_asn_pair(start: str, country_code: str | None = None,
                 asn: int | None = None) -> MLabDataFramePair:
    """Return a MLabDataFramePair for country+ASN data (optionally filtered)."""
    dl = load_parquet("downloads_by_country_asn", start)
    ul = load_parquet("uploads_by_country_asn", start)
    if country_code:
        dl = dl[dl["country_code"] == country_code]
        ul = ul[ul["country_code"] == country_code]
    if asn is not None:
        dl = dl[dl["asn"] == asn]
        ul = ul[ul["asn"] == asn]
    return MLabDataFramePair(download=dl, upload=ul)


## 1. Setting Up the IQB Cache

`IQBRemoteCache` fetches Parquet files from the M-Lab data portal and stores them locally. `IQBCache` provides the read API on top of the local cache. `IQBCalculator` computes the scores.

In [ ]:
calculator = IQBCalculator()
calculator.print_config()


## 2. IQB Score for a Single Country

We'll start with a simple example: computing the IQB score for the United States in a recent month.

In [ ]:
START_DATE = "2024-10-01"
END_DATE   = "2024-11-01"

us_pair = get_country_pair(START_DATE, "US")

print("Download data columns:", list(us_pair.download.columns))
print()
print(us_pair.download)


In [ ]:
# Extract the p95 slice (IQB report default: "top 5% performance")
us_p95 = us_pair.to_iqb_data(percentile=95)
print("US network metrics at p95:")
print(f"  Download : {us_p95.download:.1f} Mbit/s")
print(f"  Upload   : {us_p95.upload:.1f} Mbit/s")
print(f"  Latency  : {us_p95.latency:.2f} ms")
print(f"  Loss     : {us_p95.loss:.4f}")

In [ ]:
# Compute the IQB score
us_score = calculator.calculate_iqb_score(data={"m-lab": us_p95.to_dict()})
print(f"US IQB Score ({START_DATE}): {us_score:.3f}")

## 3. Comparing Countries

Let's compute scores for a range of countries and compare them.

In [ ]:
COUNTRIES = ["US", "DE", "GB", "FR", "JP", "BR", "IN", "NG", "ZA", "AU", "CA", "MX"]

results = []
for cc in COUNTRIES:
    try:
        pair = get_country_pair(START_DATE, cc)
        data = pair.to_iqb_data(percentile=95)
        score = calculator.calculate_iqb_score(data={"m-lab": data.to_dict()})
        results.append({
            "country": cc,
            "iqb_score": score,
            "download_p95": data.download,
            "upload_p95":  data.upload,
            "latency_p95": data.latency,
            "loss_p95":    data.loss,
        })
    except Exception as e:
        print(f"  Skipping {cc}: {e}")

scores_df = pd.DataFrame(results).sort_values("iqb_score", ascending=False)
scores_df


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.RdYlGn(scores_df["iqb_score"])
bars = ax.barh(scores_df["country"][::-1], scores_df["iqb_score"][::-1], color=colors[::-1])
ax.set_xlim(0, 1)
ax.set_xlabel("IQB Score (0–1, higher is better)")
ax.set_title(f"IQB Scores by country — {START_DATE} (p95 slice)")
# Add value labels
for bar, val in zip(bars, scores_df["iqb_score"][::-1]):
    ax.text(val + 0.01, bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

## 4. Per-Use-Case Breakdown

The IQB score is an average across six use cases. We can examine which use cases drive the overall score.

In [ ]:
USE_CASE_COUNTRIES = ["US", "DE", "IN", "BR"]

use_case_records = []
for cc in USE_CASE_COUNTRIES:
    try:
        pair = get_country_pair(START_DATE, cc)
        data = {"m-lab": pair.to_iqb_data(percentile=95).to_dict()}
        detailed = calculator.calculate_iqb_score(data=data, return_details=True)
        if isinstance(detailed, dict):
            for use_case, uc_score in detailed.items():
                use_case_records.append({"country": cc, "use_case": use_case, "score": uc_score})
        else:
            print(f"  {cc}: detailed output not available in this version")
    except Exception as e:
        print(f"  {cc}: {e}")

if use_case_records:
    uc_df = pd.DataFrame(use_case_records)
    uc_pivot = uc_df.pivot(index="use_case", columns="country", values="score")

    fig, ax = plt.subplots(figsize=(10, 5))
    uc_pivot.plot(kind="bar", ax=ax)
    ax.set_xlabel("Use Case")
    ax.set_ylabel("Score (0–1)")
    ax.set_ylim(0, 1.05)
    ax.set_title(f"IQB scores by use case — {START_DATE}")
    ax.legend(title="Country", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("Per-use-case breakdown not available; see manual example below.")


### Manual use-case heatmap

Even without `return_details`, we can compute per-use-case scores manually by running `calculate_iqb_score` with a custom config that isolates each use case:

In [ ]:
from iqb import IQB_CONFIG, IQBConfig

# IQB_CONFIG is the dict of use-case configs; we iterate over them
uc_names = list(IQB_CONFIG.keys())
print("Use cases in IQB_CONFIG:", uc_names)

In [ ]:
HEATMAP_COUNTRIES = ["US", "DE", "GB", "JP", "BR", "IN", "NG"]

heatmap_records = []
for cc in HEATMAP_COUNTRIES:
    try:
        pair = get_country_pair(START_DATE, cc)
        mdata = {"m-lab": pair.to_iqb_data(percentile=95).to_dict()}
        for uc_name in uc_names:
            single_uc_config = IQBConfig(use_cases={uc_name: IQB_CONFIG[uc_name]})
            single_calc = IQBCalculator(config=single_uc_config)
            uc_score = single_calc.calculate_iqb_score(data=mdata)
            heatmap_records.append({"country": cc, "use_case": uc_name, "score": uc_score})
    except Exception as e:
        print(f"  {cc}: {e}")

if heatmap_records:
    hm_df = pd.DataFrame(heatmap_records)
    hm_pivot = hm_df.pivot(index="use_case", columns="country", values="score")[HEATMAP_COUNTRIES]

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.heatmap(hm_pivot, ax=ax, cmap="RdYlGn", vmin=0, vmax=1,
                annot=True, fmt=".2f", linewidths=0.5)
    ax.set_title(f"IQB score by use case and country — {START_DATE} (p95 slice)")
    ax.set_xlabel("Country")
    ax.set_ylabel("Use Case")
    plt.tight_layout()
    plt.show()


## 5. Effect of Percentile Choice

The IQB score depends on which percentile you select. The report recommends p95 ("top 5% performance"), but p50 (median) gives a picture of the typical user experience.

In [ ]:
PERCENTILES = [5, 10, 25, 50, 75, 90, 95, 99]
PERCENTILE_COUNTRIES = ["US", "DE", "BR", "IN"]

pct_records = []
for cc in PERCENTILE_COUNTRIES:
    try:
        pair = get_country_pair(START_DATE, cc)
        for pct in PERCENTILES:
            try:
                mdata = {"m-lab": pair.to_iqb_data(percentile=pct).to_dict()}
                score = calculator.calculate_iqb_score(data=mdata)
                pct_records.append({"country": cc, "percentile": pct, "iqb_score": score})
            except ValueError:
                pass
    except Exception as e:
        print(f"  {cc}: {e}")

pct_df = pd.DataFrame(pct_records)

fig, ax = plt.subplots(figsize=(10, 5))
for cc, grp in pct_df.groupby("country"):
    ax.plot(grp["percentile"], grp["iqb_score"], marker="o", label=cc)
ax.set_xlabel("Percentile")
ax.set_ylabel("IQB Score")
ax.set_ylim(0, 1)
ax.set_title(f"IQB score sensitivity to percentile — {START_DATE}")
ax.axvline(x=95, color="gray", linestyle="--", alpha=0.5, label="p95 (report default)")
ax.axvline(x=50, color="gray", linestyle=":",  alpha=0.5, label="p50 (median)")
ax.legend()
plt.tight_layout()
plt.show()


## 6. ISP-Level IQB Scores

We can compute IQB scores for individual ISPs using the `COUNTRY_ASN` granularity.

In [ ]:
COUNTRY_FOCUS = "US"
MIN_SAMPLES   = 500

dl_asn_df = load_parquet("downloads_by_country_asn", START_DATE)
ul_asn_df = load_parquet("uploads_by_country_asn", START_DATE)

dl_asn_df = dl_asn_df[dl_asn_df["country_code"] == COUNTRY_FOCUS]
ul_asn_df = ul_asn_df[ul_asn_df["country_code"] == COUNTRY_FOCUS]

reliable_asns = dl_asn_df[dl_asn_df["sample_count"] >= MIN_SAMPLES]["asn"].unique()
print(f"{len(reliable_asns)} reliable ASNs in {COUNTRY_FOCUS}")


In [ ]:
from iqb.cache.mlab import MLabDataFramePair

asn_score_records = []
for asn in reliable_asns:
    try:
        dl_row = dl_asn_df[dl_asn_df["asn"] == asn]
        ul_row = ul_asn_df[ul_asn_df["asn"] == asn]
        if dl_row.empty or ul_row.empty:
            continue
        pair = MLabDataFramePair(download=dl_row, upload=ul_row)
        mdata = {"m-lab": pair.to_iqb_data(percentile=95).to_dict()}
        score = calculator.calculate_iqb_score(data=mdata)
        asn_score_records.append({
            "asn": asn,
            "iqb_score": score,
            "sample_count": int(dl_row.iloc[0]["sample_count"]),
        })
    except Exception:
        pass

asn_scores = pd.DataFrame(asn_score_records).sort_values("iqb_score", ascending=False)
print(f"Scored {len(asn_scores)} ASNs")
asn_scores.head(10)

In [ ]:
top_asn = asn_scores.nlargest(15, "iqb_score")

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.RdYlGn(top_asn["iqb_score"])
ax.barh(top_asn["asn"].astype(str)[::-1], top_asn["iqb_score"][::-1], color=colors[::-1])
ax.set_xlim(0, 1)
ax.set_xlabel("IQB Score")
ax.set_title(f"Top 15 ASNs by IQB score — {COUNTRY_FOCUS} ({START_DATE})")
plt.tight_layout()
plt.show()

## 7. Time Series of IQB Scores

Looping over months lets us track how a country's IQB score evolves over time.

In [ ]:
months = [
    ("2024-01-01", "2024-02-01"), ("2024-02-01", "2024-03-01"),
    ("2024-03-01", "2024-04-01"), ("2024-04-01", "2024-05-01"),
    ("2024-05-01", "2024-06-01"), ("2024-06-01", "2024-07-01"),
    ("2024-07-01", "2024-08-01"), ("2024-08-01", "2024-09-01"),
    ("2024-09-01", "2024-10-01"), ("2024-10-01", "2024-11-01"),
    ("2024-11-01", "2024-12-01"), ("2024-12-01", "2025-01-01"),
]
TS_COUNTRIES = ["US", "DE", "BR", "IN"]

ts_records = []
for start, _ in months:
    for cc in TS_COUNTRIES:
        try:
            pair = get_country_pair(start, cc)
            mdata = {"m-lab": pair.to_iqb_data(percentile=95).to_dict()}
            score = calculator.calculate_iqb_score(data=mdata)
            ts_records.append({"month": pd.to_datetime(start), "country": cc, "iqb_score": score})
        except Exception:
            pass

ts_df = pd.DataFrame(ts_records)
print(f"Time series: {ts_df['month'].nunique()} months × {ts_df['country'].nunique()} countries")


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for cc, grp in ts_df.groupby("country"):
    grp = grp.sort_values("month")
    ax.plot(grp["month"], grp["iqb_score"], marker="o", label=cc)

ax.set_ylim(0, 1)
ax.set_ylabel("IQB Score")
ax.set_title("IQB Score over time — selected countries (p95 slice)")
ax.legend(title="Country")
ax.xaxis.set_major_locator(mticker.MaxNLocator(10))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 8. Customizing IQB Thresholds

The IQB framework is designed to be parameterizable. You can modify the quality thresholds to answer questions like "what if we used more demanding requirements for video conferencing?"

In [ ]:
from iqb import IQB_DEFAULT_CONFIG, IQBConfig, IQBConfigUseCase, IQBConfigNetworkRequirement

# Inspect the default video conferencing thresholds
vc_config = IQB_DEFAULT_CONFIG.use_cases.get("video_conferencing")
if vc_config:
    print("Default video conferencing requirements:")
    for req in vc_config.requirements:
        print(f"  {req}")

In [ ]:
pair = get_country_pair(START_DATE, "US")
mdata = {"m-lab": pair.to_iqb_data(percentile=95).to_dict()}

default_score = calculator.calculate_iqb_score(data=mdata)
print(f"Default IQB score (US, p95): {default_score:.3f}")

print("\nSee IQBConfig documentation for threshold customization.")


---

## Summary

The IQB score provides a compact, multi-dimensional view of internet quality that goes beyond speed alone. Key takeaways:

- The score ranges 0–1 and aggregates six use-case requirements
- The percentile choice matters: p95 reflects "top performers" while p50 reflects the typical user
- City, subdivision, and ASN granularities allow drilling into geographic and ISP-level differences
- Custom threshold configurations make it possible to explore sensitivity to specific requirements

### Useful resources

- [M-Lab IQB GitHub](https://github.com/m-lab/iqb) — library source, pipeline docs, and analysis notebooks
- [IQB Framework Report](https://www.measurementlab.net/publications/IQB_report_2025.pdf)
- [IQB Executive Summary](https://www.measurementlab.net/publications/IQB_executive_summary_2025.pdf)
- [01-exploring-iqb-data.ipynb](01-exploring-iqb-data.ipynb) — raw Monthly Stats exploration without the iqb library